In [1]:
# ==============================================================================
#           NOTEBOOK DE TESTE E AUDITORIA PARA O MÓDULO DE ORÁCULOS
# ==============================================================================
# Este notebook importa a fábrica de oráculos e executa uma série de testes
# de auditoria para validar a chamada e o parsing de diferentes provedores de LLM.

import os
import sys
import json
from dotenv import load_dotenv

# --- 1. CONFIGURAÇÃO DO AMBIENTE ---
# Adiciona o diretório raiz do projeto ao path do sistema.
try:
    from activetextclassification.oraculo.oracles import get_oracle
    from activetextclassification.oraculo.prompts import PROMPTS_ORACULO # Importa os prompts para visualização
    print("Módulo 'oracles' e fábrica 'get_oracle' importados com sucesso.")
except ImportError:
    print("Módulo não encontrado. Tentando ajustar o sys.path...")
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if project_root not in sys.path:
        print(f"Adicionando diretório raiz ao path: {project_root}")
        sys.path.insert(0, project_root)
    from activetextclassification.oraculo.oracles import get_oracle
    from activetextclassification.oraculo.prompts import PROMPTS_ORACULO # Importa os prompts para visualização
    print("Módulo 'oracles' importado com sucesso após ajuste do path.")


# --- 2. FUNÇÃO DE TESTE MODULAR ---

def run_oracle_audit(test_config):
    """
    Função reutilizável para auditar a chamada e o parsing de um oráculo específico.
    
    Args:
        test_config (dict): Dicionário com a configuração do modelo a ser testado.
    """
    print("\n" + "="*25 + f" INICIANDO AUDITORIA PARA: {test_config['model_name']} " + "="*25)
    load_dotenv()

    # Dados de teste padrão
    test_descriptions = [
        {"text_sample": "Leite Cond. Mococa TP 395G", "true_label_sample": "leite condensado"},
        {"text_sample": "SABAO EM PO OMO LAV PERF CX 1.6KG", "true_label_sample": "lava roupas"}
    ]
    test_labels = ["leite condensado", "lava roupas", "acucar", "biscoito", "_RARE_"]
    labels_str_for_test = json.dumps(test_labels, ensure_ascii=False)

    print(f"\nConfiguração de Teste: {json.dumps(test_config, indent=2)}")

    try:
        # Criação do Oráculo
        print("\n1. Criando instância do oráculo...")
        # A instância é criada, mas a consulta ainda não foi feita
        oracle_instance = get_oracle(test_config, labels_str_for_test)
        print(f"Oráculo '{oracle_instance.__class__.__name__}' criado com sucesso.")

        # --- NOVA SEÇÃO VERBOSA ---
        # Monta o prompt exatamente como a classe do oráculo faria para que possamos visualizá-lo.
        print("\n2. Visualizando o prompt que será enviado ao LLM...")
        batch_input_list = [{"id": i + 1, "descricao": item['text_sample']} for i, item in enumerate(test_descriptions)]
        descricoes_json = json.dumps(batch_input_list, indent=2, ensure_ascii=False)
        
        # Acessa o template de prompt usado pela instância do oráculo
        full_prompt_to_send = oracle_instance.prompt_template.format(
            lista_categorias_str=labels_str_for_test, 
            descricoes_lote_json=descricoes_json
        )
        print("-" * 70)
        print(full_prompt_to_send)
        print("-" * 70)
        # --- FIM DA SEÇÃO VERBOSA ---

        # Execução da Consulta
        print("\n3. Executando a consulta (query)...")
        results, call_log = oracle_instance.query(test_descriptions)

        # Auditoria dos Resultados
        print("\n4. Auditoria dos Resultados Recebidos:")
        print("\n--- Log da Chamada de API ---")
        print(json.dumps(call_log, indent=2))

        print("\n--- Resultados Processados (Parsing) ---")
        for res in results:
            print(json.dumps(res, indent=2, ensure_ascii=False))
            print("-" * 20)

    except (ImportError, ValueError, RuntimeError) as e:
        print(f"\nERRO CRÍTICO NO TESTE para {test_config['model_name']}: {e}")
        import traceback
        traceback.print_exc()
    
    print("="*35 + f" FIM DA AUDITORIA PARA: {test_config['model_name']} " + "="*35)


# --- 3. EXECUÇÃO DOS TESTES ---

# Defina aqui a lista de configurações que você deseja testar.
configs_to_test = [
    # {
    #     "model_name": "qwen2.5",
    #     "temperature": 0.1,
    #     "prompt_version_key": "v1" # Use a chave definida no seu prompts.py
    # },
    # {
    #     "model_name": "gemma3",
    #     "temperature": 0.1,
    #     "prompt_version_key": "v1"
    # },
    # {
    #     "model_name": "gemini-1.5-flash-latest",
    #     "temperature": 0.1,
    #     "prompt_version_key": "v1"
    # },
    {
        "model_name": "gpt-4o-mini",
        "temperature": 0.1,
        "prompt_version_key": "v1"
    }
]

# Itera sobre cada configuração e executa a função de auditoria
for config in configs_to_test:
    run_oracle_audit(config)

print("\n--- TODOS OS TESTES DE AUDITORIA FORAM CONCLUÍDOS ---")

Módulo 'oracles' e fábrica 'get_oracle' importados com sucesso.

========================= INICIANDO AUDITORIA PARA: gemini-1.5-flash-latest =========================

Configuração de Teste: {
  "model_name": "gemini-1.5-flash-latest",
  "temperature": 0.1,
  "prompt_version_key": "v1"
}

1. Criando instância do oráculo...
Cliente Google GenAI configurado com sucesso.
Oráculo 'GoogleOracle' criado com sucesso.

2. Visualizando o prompt que será enviado ao LLM...
----------------------------------------------------------------------

Você é um especialista em catalogação de produtos e processamento de linguagem natural, operando em modo de alta eficiência. 
Sua tarefa é receber um array JSON de descrições de produtos e, para cada um, realizar uma análise detalhada, 
retornando os resultados em um formato JSON estruturado.

**Formato da Entrada:**
Você receberá um array JSON onde cada objeto contém um `id` único e uma `descricao`.

**Formato da Saída OBRIGATÓRIO:**
Sua resposta DEVE SER 